# 12 - Análisis por subgrupos y ablación

Este notebook cubre dos análisis exploratorios solicitados en la devolución: rendimiento por edad/fuente del diagnóstico y ablación de características del baseline (forma, color y ambas). No evalúa el test.

**Limitación importante:** en la validación disponible todos los melanomas tienen diagnóstico histopatológico; seguimiento y consenso solo incluyen nevos. Por eso no se puede estimar recall de melanoma fuera del grupo histopatológico ni separar plenamente la fuente del diagnóstico de la clase. Los tamaños por subgrupo también son pequeños; interpretar las métricas descriptivamente.

## 1. Preparar entorno, modelos y particiones

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from torch.utils.data import DataLoader

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.dataset import LesionDataset, get_transforms
from src.entrenamiento import probabilidades
from src.experimento_class_weight import (
    IMAGES,
    UMBRALES,
    UMBRAL_RECALL,
    metricas,
    preparar_particiones,
    seleccionar_umbral,
)
from src.modelo import crear_modelo
from src.utils import SEED

RAW_METADATA = ROOT / "data" / "raw" / "HAM10000_metadata.csv"
OLD_TEST = ROOT / "data" / "processed" / "test.csv"
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "models"
REPORT_PATH = ROOT / "reports" / "subgrupos_ablacion.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

Dispositivo: cpu


## 2. Rendimiento por edad y fuente del diagnóstico

Se usan los modelos experimentales y la validación sin lesiones del test anterior. Para edad se describen dos grupos (`<50` y `≥50`); las edades faltantes se excluyen y se cuentan. El umbral es el seleccionado globalmente para cada modelo, no uno ajustado dentro de cada subgrupo.

In [2]:
train_df, val_df = preparar_particiones(RAW_METADATA, OLD_TEST)
old_test_lesions = set(pd.read_csv(OLD_TEST)["lesion_id"])
assert set(val_df["lesion_id"]).isdisjoint(old_test_lesions)

val_loader = DataLoader(
    LesionDataset(val_df, get_transforms(entrenamiento=False), image_dir=IMAGES),
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

probabilidades_modelos = {}
umbrales_modelos = {}
y_true = None
for nombre, checkpoint in {
    "class_weight": MODELS / "cnn_resnet18_class_weight.pt",
    "submuestreo": MODELS / "cnn_resnet18_subsampled_control.pt",
}.items():
    if not checkpoint.exists():
        raise FileNotFoundError(f"No se encontró el checkpoint: {checkpoint}")
    modelo = crear_modelo(preentrenado=False).to(device)
    modelo.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))
    probs, etiquetas = probabilidades(modelo, val_loader, device)
    if y_true is None:
        y_true = etiquetas
    elif not np.array_equal(y_true, etiquetas):
        raise ValueError("Los modelos no tienen las mismas etiquetas de validación.")
    umbral = seleccionar_umbral(y_true, probs)
    if umbral is None:
        raise ValueError(f"{nombre} no alcanza el recall mínimo en la grilla de umbrales.")
    probabilidades_modelos[nombre] = probs
    umbrales_modelos[nombre] = umbral
    del modelo

resultados_subgrupos = []
val_predicciones = val_df.copy().reset_index(drop=True)
val_predicciones["y_true"] = y_true
val_predicciones["grupo_edad"] = pd.cut(
    val_predicciones["age"],
    bins=[-np.inf, 50, np.inf],
    labels=["<50", "≥50"],
    right=False,
)

for nombre, probs in probabilidades_modelos.items():
    val_predicciones["probabilidad"] = probs
    val_predicciones["prediccion"] = (probs >= umbrales_modelos[nombre]).astype(int)
    for tipo_grupo, columna in [("edad", "grupo_edad"), ("fuente_diagnostico", "dx_type")]:
        for grupo, datos in val_predicciones.dropna(subset=[columna]).groupby(columna, observed=True):
            positivos = int(datos["y_true"].sum())
            negativos = int((datos["y_true"] == 0).sum())
            falsos_negativos = int(((datos["y_true"] == 1) & (datos["prediccion"] == 0)).sum())
            falsos_positivos = int(((datos["y_true"] == 0) & (datos["prediccion"] == 1)).sum())
            fila = {
                "modelo": nombre,
                "tipo_grupo": tipo_grupo,
                "grupo": str(grupo),
                "n": len(datos),
                "n_melanomas": positivos,
                "n_nevos": negativos,
                "umbral_global": umbrales_modelos[nombre],
                "recall_melanoma": (
                    float(recall_score(datos["y_true"], datos["prediccion"]))
                    if positivos else None
                ),
                "especificidad_nevo": (
                    float(((datos["y_true"] == 0) & (datos["prediccion"] == 0)).sum() / negativos)
                    if negativos else None
                ),
                "f1_macro": (
                    float(f1_score(datos["y_true"], datos["prediccion"], average="macro"))
                    if positivos and negativos else None
                ),
                "falsos_negativos": falsos_negativos,
                "falsos_positivos": falsos_positivos,
            }
            resultados_subgrupos.append(fila)

tabla_subgrupos = pd.DataFrame(resultados_subgrupos)
print("Edades faltantes excluidas del análisis por edad:", int(val_df["age"].isna().sum()))
display(tabla_subgrupos)

Edades faltantes excluidas del análisis por edad: 9


,modelo,tipo_grupo,grupo,n,n_melanomas,n_nevos,umbral_global,recall_melanoma,especificidad_nevo,f1_macro,falsos_negativos,falsos_positivos
0,class_weight,edad,<50,579,25,554,0.5,0.800000,0.888087,0.655041,5,62
1,class_weight,edad,≥50,567,72,495,0.5,0.902778,0.832323,0.746220,7,83
2,class_weight,fuente_diagnostico,consensus,71,0,71,0.5,NaN,0.690141,NaN,0,22
3,class_weight,fuente_diagnostico,follow_up,726,0,726,0.5,NaN,0.988981,NaN,0,8
4,class_weight,fuente_diagnostico,histo,358,98,260,0.5,0.877551,0.550000,0.630293,12,117
5,submuestreo,edad,<50,579,25,554,0.6,0.680000,0.855596,0.596873,8,80
6,submuestreo,edad,≥50,567,72,495,0.6,0.916667,0.830303,0.747955,6,84
7,submuestreo,fuente_diagnostico,consensus,71,0,71,0.6,NaN,0.619718,NaN,0,27
8,submuestreo,fuente_diagnostico,follow_up,726,0,726,0.6,NaN,0.995868,NaN,0,3
9,submuestreo,fuente_diagnostico,histo,358,98,260,0.6,0.857143,0.476923,0.575709,14,136


## 3. Ablación del baseline: forma y color

Se comparan Random Forest con características geométricas, solo color y todas juntas. Se usa el split original de la CNN: 785 imágenes de entrenamiento y 197 de validación. Las características se filtran por `image_id`; el test no se usa.

In [3]:
features = pd.read_csv(PROCESSED / "features_train.csv")
train_balanced = pd.read_csv(PROCESSED / "train_cnn.csv")
val_balanced = pd.read_csv(PROCESSED / "val_cnn.csv")

features_train = features.merge(
    train_balanced[["image_id", "etiqueta"]],
    on=["image_id", "etiqueta"],
    validate="one_to_one",
)
features_val = features.merge(
    val_balanced[["image_id", "etiqueta"]],
    on=["image_id", "etiqueta"],
    validate="one_to_one",
)
assert len(features_train) == len(train_balanced) == 785
assert len(features_val) == len(val_balanced) == 197
assert set(features_train.image_id).isdisjoint(features_val.image_id)

geometricas = ["area", "area_rel", "perimetro", "circularidad", "solidez", "asimetria"]
color = [c for c in features.columns if c.startswith(("hsv", "lab"))]
conjuntos = {
    "solo_forma": geometricas,
    "solo_color": color,
    "forma_y_color": geometricas + color,
}
assert len(color) == 12

resultados_ablacion = {}
for nombre, columnas in conjuntos.items():
    rf = RandomForestClassifier(n_estimators=300, random_state=SEED)
    rf.fit(features_train[columnas], features_train["etiqueta"])
    probs = rf.predict_proba(features_val[columnas])[:, 1]
    reales = features_val["etiqueta"].to_numpy()
    umbral = seleccionar_umbral(reales, probs)
    resultados_ablacion[nombre] = {
        "caracteristicas": columnas,
        "n_caracteristicas": len(columnas),
        "umbral_seleccionado": umbral,
        "metricas_umbral_0_5": metricas(reales, probs, 0.5),
        "metricas_umbral_seleccionado": (
            metricas(reales, probs, umbral) if umbral is not None else None
        ),
    }

tabla_ablacion = pd.DataFrame(
    {
        nombre: resultado["metricas_umbral_seleccionado"]
        for nombre, resultado in resultados_ablacion.items()
        if resultado["metricas_umbral_seleccionado"] is not None
    }
).T
display(tabla_ablacion[["umbral", "recall_melanoma", "f1_macro", "especificidad", "falsos_negativos", "falsos_positivos"]])
print("Umbrales seleccionados:", {k: v["umbral_seleccionado"] for k, v in resultados_ablacion.items()})

,umbral,recall_melanoma,f1_macro,especificidad,falsos_negativos,falsos_positivos
solo_color,0.6,0.857143,0.78587,0.717172,14,28
forma_y_color,0.5,0.897959,0.795048,0.69697,10,30


Umbrales seleccionados: {'solo_forma': None, 'solo_color': 0.6, 'forma_y_color': 0.5}


## 4. Guardar resultados

Estos análisis usan validaciones para describir patrones y seleccionar umbrales; sus resultados son exploratorios. En las categorías de fuente sin melanomas solo se interpreta la especificidad de nevos, no recall ni F1.

In [4]:
reporte = {
    "evaluacion": "validaciones exploratorias; no se uso el test",
    "semilla": SEED,
    "subgrupos_cnn": {
        "particion": "validacion del experimento class_weight vs submuestreo",
        "umbrales_globales": umbrales_modelos,
        "edad_corte": "<50 vs >=50",
        "edades_faltantes_excluidas": int(val_df["age"].isna().sum()),
        "dx_type_por_clase": pd.crosstab(val_df["dx_type"], val_df["dx"]).to_dict(),
        "nota_fuente": "todos los melanomas de validacion son histopatologicos; seguimiento y consenso solo contienen nevos",
        "resultados": resultados_subgrupos,
    },
    "ablacion_baseline": {
        "particion": "train_cnn/val_cnn original; test no usado",
        "train": len(features_train),
        "validacion": len(features_val),
        "regla_umbral": {
            "grilla": list(UMBRALES),
            "recall_minimo": UMBRAL_RECALL,
            "criterio": "mayor F1 macro entre umbrales elegibles",
        },
        "resultados": resultados_ablacion,
    },
}
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text(
    json.dumps(reporte, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(f"Reporte guardado en {REPORT_PATH}")

Reporte guardado en c:\Trabajo final\reports\subgrupos_ablacion.json
